In [1]:
import pandas as pd
import numpy as np
import warnings

# Tắt cảnh báo không cần thiết của pandas
warnings.filterwarnings('ignore')

# 1. ĐỌC DỮ LIỆU
file_path = 'YTD_Actuals_Ledger.csv'
df_raw = pd.read_csv(file_path)

print(f"Tổng số dòng dữ liệu thô ban đầu: {len(df_raw)}")

Tổng số dòng dữ liệu thô ban đầu: 6860


In [2]:
# 2. LOẠI BỎ DÒNG TRỐNG (Blank transaction rows)
# Giả định dòng trống là dòng mà Txn_ID và Amount_GBP đều NaN
blank_rows_mask = df_raw['Txn_ID'].isna() | df_raw['Amount_GBP'].isna()
blank_rows_count = blank_rows_mask.sum()
print(f"Số dòng trống bị loại bỏ (Excluded): {blank_rows_count}")

# Tạo dataframe làm sạch cơ bản
df_clean = df_raw[~blank_rows_mask].copy()

# Ép kiểu dữ liệu cho cột Amount và Txn_Date để tính toán
df_clean['Amount_GBP'] = pd.to_numeric(df_clean['Amount_GBP'], errors='coerce')
df_clean['Txn_Date'] = pd.to_datetime(df_clean['Txn_Date'], errors='coerce')

print(f"Số dòng dữ liệu hợp lệ sau khi bỏ trống: {len(df_clean)}")

Số dòng trống bị loại bỏ (Excluded): 30
Số dòng dữ liệu hợp lệ sau khi bỏ trống: 6830


In [4]:
# 3. ĐÁNH DẤU GIAO DỊCH TRÙNG (Exact duplicate transactions)
# Đánh dấu tất cả các dòng bị trùng (keep=False)
df_clean['Is_Duplicate'] = df_clean.duplicated(keep=False)

# Đếm số cặp/dòng bị trùng 
duplicate_excess_count = df_clean.duplicated(keep='first').sum()
duplicate_total_flagged = df_clean['Is_Duplicate'].sum()
print(f"Số giao dịch dư thừa do trùng lặp (cần review/xóa): {duplicate_excess_count}")

Số giao dịch dư thừa do trùng lặp (cần review/xóa): 25


In [12]:
df_dedup = df_clean.drop_duplicates(keep='first')

In [13]:
# ============================================================
# BƯỚC 1: TRÍCH XUẤT CÁC ACCOUNT_CODE THỰC TẾ CÓ TRONG FILE
# ============================================================

# Chuẩn hóa Account_Code: bỏ khoảng trắng, bỏ đuôi .0 nếu bị đọc thành float
def normalize_code(x):
    if pd.isna(x):
        return ''
    s = str(x).strip()
    if s.endswith('.0'):
        s = s[:-2]
    return s

df_flagged = df_dedup.copy()
df_flagged['Account_Code'] = df_flagged['Account_Code'].apply(normalize_code)

print("=" * 60)
print("TẤT CẢ GIÁ TRỊ ACCOUNT_CODE CÓ TRONG FILE (value_counts)")
print("=" * 60)
account_counts = df_flagged['Account_Code'].value_counts(dropna=False)
print(account_counts.to_string())

print(f"\n→ Tổng số Account_Code khác nhau trong file: {df_flagged['Account_Code'].nunique()}")
print(f"→ Số dòng có Account_Code rỗng: {(df_flagged['Account_Code'] == '').sum()}")

TẤT CẢ GIÁ TRỊ ACCOUNT_CODE CÓ TRONG FILE (value_counts)
Account_Code
4030    482
4040    480
6010    459
6000    455
7000    455
5010    451
4000    442
5000    440
4010    439
4020    431
6110    430
6100    412
6200    399
6300    318
6500    125
6600    119
6210    116
6700    112
6400    111
9000     40
6999     34
5099     31
7100     24

→ Tổng số Account_Code khác nhau trong file: 23
→ Số dòng có Account_Code rỗng: 0


In [15]:
# ============================================================
# KHỐI 4 (NEW): MAPPING + NORMALIZE DESCRIPTION + VERIFY SIGN
# Nguồn: df_dedup (đã bỏ blank + xóa duplicate)
# ============================================================

# ---------- 4.1. Chuẩn hóa Account_Code ----------
def normalize_code(x):
    if pd.isna(x):
        return ''
    s = str(x).strip()
    if s.endswith('.0'):
        s = s[:-2]
    return s

# Làm việc trên df_dedup (KHÔNG phải df_clean nữa)
df_dedup['Account_Code'] = df_dedup['Account_Code'].apply(normalize_code)
df_dedup['Account_Code_Padded'] = df_dedup['Account_Code'].str.zfill(4)

# ---------- 4.2. Mapping table (23 mã CoA) ----------
mapping_dict = {
    '4000': ('Food & Beverage Revenue',     'Revenue'),
    '4010': ('Retail Merchandise Revenue',  'Revenue'),
    '4020': ('Entertainment Revenue',       'Revenue'),
    '4030': ('Membership Revenue',          'Revenue'),
    '4040': ('Events & Hire Revenue',       'Revenue'),
    '5000': ('Cost of Goods Sold - F&B',    'COGS'),
    '5010': ('Cost of Goods Sold - Retail', 'COGS'),
    '5099': ('Cost of Goods Sold - Other',  'COGS'),
    '6000': ('Wages & Salaries',            'OPEX_Wages'),
    '6010': ('Employer NI & Pension',       'OPEX_Wages'),
    '6100': ('Property Rent',               'OPEX_Fixed'),
    '6110': ('Business Rates',              'OPEX_Fixed'),
    '6500': ('IT & Software Licences',      'OPEX_Fixed'),
    '6600': ('Insurance',                   'OPEX_Fixed'),
    '6200': ('Utilities - Electricity',     'OPEX_Variable'),
    '6210': ('Utilities - Gas/Water',       'OPEX_Variable'),
    '6300': ('Marketing & Promotions',      'OPEX_Variable'),
    '6400': ('Repairs & Maintenance',       'OPEX_Variable'),
    '6700': ('Professional Fees',           'OPEX_Variable'),
    '7000': ('Depreciation',                'Non-Cash'),
    '7100': ('Bank Interest',               'Finance'),
    '6999': ('Other Operating Expenses',    'OPEX_Variable'),
    '9000': ('Other Income/Expense',        'Finance'),
}

# ---------- 4.3. Map Category + Name ----------
df_dedup['Account_Name_Mapped'] = df_dedup['Account_Code_Padded'].map(
    lambda c: mapping_dict.get(c, ('UNMAPPED', 'UNMAPPED'))[0]
)
df_dedup['P&L_Category'] = df_dedup['Account_Code_Padded'].map(
    lambda c: mapping_dict.get(c, ('UNMAPPED', 'UNMAPPED'))[1]
)
df_dedup['Is_Unmapped_Account'] = df_dedup['P&L_Category'] == 'UNMAPPED'

# ---------- 4.4. Normalize Description ----------
# Chuẩn hóa format: lowercase → capitalize first letter
def normalize_description(desc):
    if pd.isna(desc) or str(desc).strip() == '':
        return ''
    s = str(desc).strip().lower()
    return s[0].upper() + s[1:] if len(s) > 0 else ''

df_dedup['Description_Clean'] = df_dedup['Description'].apply(normalize_description)

# Ghi đè bằng tên chuẩn từ CoA (nhất quán 100%)
df_dedup['Description_Final'] = df_dedup['Account_Name_Mapped']

# Flag nếu Description gốc khác CoA name (Informational)
df_dedup['Description_Mismatch'] = (
    df_dedup['Description_Clean'].str.lower().str.strip()
    != df_dedup['Account_Name_Mapped'].str.lower().str.strip()
) & (~df_dedup['Is_Unmapped_Account'])

# ---------- 4.5. Verify SIGN CONVENTION ----------
def expected_sign(code):
    return 1 if code.startswith('4') else -1

df_dedup['Expected_Sign'] = df_dedup['Account_Code_Padded'].apply(expected_sign)
df_dedup['Actual_Sign']   = np.sign(df_dedup['Amount_GBP'])
df_dedup['Sign_Violation'] = (
    (df_dedup['Actual_Sign'] != 0)
    & (df_dedup['Actual_Sign'] != df_dedup['Expected_Sign'])
)

print("=" * 70)
print("KIỂM TRA SIGN CONVENTION (trên df_dedup)")
print("=" * 70)
sign_check = df_dedup.groupby('Account_Code_Padded').agg(
    So_dong=('Amount_GBP', 'size'),
    Tong_tien=('Amount_GBP', 'sum'),
    Min=('Amount_GBP', 'min'),
    Max=('Amount_GBP', 'max'),
    So_dong_sai_dau=('Sign_Violation', 'sum'),
).reset_index()
sign_check['Expected'] = sign_check['Account_Code_Padded'].apply(
    lambda c: '+' if c.startswith('4') else '-'
)
print(sign_check.to_string(index=False))
print(f"\n→ Tổng số dòng SAI DẤU: {df_dedup['Sign_Violation'].sum()}")

# ---------- 4.6. Phân loại lỗi ----------
critical_unmapped = df_dedup['Is_Unmapped_Account']
critical_bad_date = df_dedup['Txn_Date'].isna()
critical_bad_amt  = df_dedup['Amount_GBP'].isna()
critical_sign     = df_dedup['Sign_Violation']

df_dedup['Error_Critical'] = critical_unmapped | critical_bad_date | critical_bad_amt | critical_sign
df_dedup['Critical_Reason'] = np.select(
    [critical_unmapped, critical_bad_date, critical_bad_amt, critical_sign],
    ["Unmapped Account", "Invalid Date", "Invalid Amount", "Wrong Sign"],
    default=""
)

# Warning: thiếu Department ở Revenue/COGS
is_rev_cogs  = df_dedup['Account_Code_Padded'].str.startswith(('4', '5'))
missing_dept = (
    df_dedup['Department'].isna()
    | df_dedup['Department'].astype(str).str.strip().isin(['', 'N/A', 'NA', '-', 'None', 'null', 'Unknown'])
)
df_dedup['Error_Warning'] = is_rev_cogs & missing_dept
df_dedup['Warning_Reason'] = np.where(df_dedup['Error_Warning'], "Missing Department (Rev/COGS)", "")

# Info: Description khác CoA name
df_dedup['Error_Info'] = df_dedup['Description_Mismatch']
df_dedup['Info_Reason'] = np.where(df_dedup['Error_Info'], "Description differs from CoA name", "")

# ---------- 4.7. Tổng hợp ----------
print("\n" + "=" * 70)
print("KẾT QUẢ PHÂN LOẠI LỖI (df_dedup)")
print("=" * 70)
print(f"Total rows                : {len(df_dedup)}")
print(f"Unmapped Account          : {critical_unmapped.sum()}")
print(f"Invalid Date              : {critical_bad_date.sum()}")
print(f"Invalid Amount            : {critical_bad_amt.sum()}")
print(f"Wrong Sign (Critical)     : {critical_sign.sum()}")
print(f"→ Total Critical          : {df_dedup['Error_Critical'].sum()}")
print(f"→ Total Warning           : {df_dedup['Error_Warning'].sum()}")
print(f"→ Total Informational     : {df_dedup['Error_Info'].sum()}")

print("\n" + "=" * 70)
print("PHÂN BỔ THEO P&L CATEGORY (df_dedup)")
print("=" * 70)
pl_summary = df_dedup.groupby('P&L_Category').agg(
    So_dong=('Amount_GBP', 'size'),
    Tong_tien_GBP=('Amount_GBP', 'sum'),
).reset_index()
print(pl_summary.to_string(index=False))

print("\n" + "=" * 70)
print("KIỂM TRA EBITDA LOGIC (sign sanity)")
print("=" * 70)
rev  = df_dedup[df_dedup['P&L_Category'] == 'Revenue']['Amount_GBP'].sum()
cogs = df_dedup[df_dedup['P&L_Category'] == 'COGS']['Amount_GBP'].sum()
print(f"Revenue (dương)     : {rev:>15,.2f}")
print(f"COGS (âm)           : {cogs:>15,.2f}")
print(f"Gross Profit        : {rev + cogs:>15,.2f}   ← phải DƯƠNG mới đúng")

KIỂM TRA SIGN CONVENTION (trên df_dedup)
Account_Code_Padded  So_dong   Tong_tien       Min      Max  So_dong_sai_dau Expected
               4000      442  7614254.67   6958.59 76115.70                0        +
               4010      439  2494868.09   2216.04 20518.82                0        +
               4020      431  4718686.81   4041.16 46088.83                0        +
               4030      482  1696254.09   1955.68 11852.09                0        +
               4040      480  1116776.47    769.79  7635.59                0        +
               5000      440 -2563544.47 -16472.68 -2399.43                0        -
               5010      451 -1447606.48 -10396.70 -1305.65                0        -
               5099       31  -116922.99 -14631.24  -406.08                0        -
               6000      455 -4578991.02 -35176.05 -5085.41                0        -
               6010      459  -632561.20  -4441.12  -689.25                0        -
             

In [17]:
# ============================================================
# KHỐI 4 (NEW): CHUẨN HÓA DESCRIPTION + MAPPING + VERIFY MATCH
# Nguồn: df_dedup (từ Khối 3)
# ============================================================

# ---------- 4.1. Chuẩn hóa Account_Code ----------
def normalize_code(x):
    if pd.isna(x):
        return ''
    s = str(x).strip()
    if s.endswith('.0'):
        s = s[:-2]
    return s

df_dedup['Account_Code'] = df_dedup['Account_Code'].apply(normalize_code)
df_dedup['Account_Code_Padded'] = df_dedup['Account_Code'].str.zfill(4)

# ---------- 4.2. Mapping table (23 mã CoA) ----------
mapping_dict = {
    '4000': ('Food & Beverage Revenue',     'Revenue'),
    '4010': ('Retail Merchandise Revenue',  'Revenue'),
    '4020': ('Entertainment Revenue',       'Revenue'),
    '4030': ('Membership Revenue',          'Revenue'),
    '4040': ('Events & Hire Revenue',       'Revenue'),
    '5000': ('Cost of Goods Sold - F&B',    'COGS'),
    '5010': ('Cost of Goods Sold - Retail', 'COGS'),
    '5099': ('Cost of Goods Sold - Other',  'COGS'),
    '6000': ('Wages & Salaries',            'OPEX_Wages'),
    '6010': ('Employer NI & Pension',       'OPEX_Wages'),
    '6100': ('Property Rent',               'OPEX_Fixed'),
    '6110': ('Business Rates',              'OPEX_Fixed'),
    '6500': ('IT & Software Licences',      'OPEX_Fixed'),
    '6600': ('Insurance',                   'OPEX_Fixed'),
    '6200': ('Utilities - Electricity',     'OPEX_Variable'),
    '6210': ('Utilities - Gas/Water',       'OPEX_Variable'),
    '6300': ('Marketing & Promotions',      'OPEX_Variable'),
    '6400': ('Repairs & Maintenance',       'OPEX_Variable'),
    '6700': ('Professional Fees',           'OPEX_Variable'),
    '7000': ('Depreciation',                'Non-Cash'),
    '7100': ('Bank Interest',               'Finance'),
    '6999': ('Other Operating Expenses',    'OPEX_Variable'),
    '9000': ('Other Income/Expense',        'Finance'),
}

# ---------- 4.3. Map Category + Name từ Account_Code ----------
df_dedup['Account_Name_Mapped'] = df_dedup['Account_Code_Padded'].map(
    lambda c: mapping_dict.get(c, ('UNMAPPED', 'UNMAPPED'))[0]
)
df_dedup['P&L_Category'] = df_dedup['Account_Code_Padded'].map(
    lambda c: mapping_dict.get(c, ('UNMAPPED', 'UNMAPPED'))[1]
)
df_dedup['Is_Unmapped_Account'] = df_dedup['P&L_Category'] == 'UNMAPPED'

# ---------- 4.4. Chuẩn hóa Description: lowercase → Title Case (mỗi chữ đầu) ----------
# Ví dụ: "FOOD & BEVERAGE REVENUE" -> "Food & Beverage Revenue"
#        "wages & salaries"         -> "Wages & Salaries"
#        "cost of goods sold - f&b" -> "Cost Of Goods Sold - F&B"
df_dedup['Description_Original'] = df_dedup['Description'].astype(str)
df_dedup['Description_Clean'] = (
    df_dedup['Description_Original']
    .str.lower()                # lowercase toàn bộ
    .str.title()                # capitalize chữ cái đầu MỖI từ
    .str.strip()
)

# ---------- 4.5. Đối chiếu Description_Clean vs Account_Name_Mapped ----------
# So sánh không phân biệt hoa/thường, không phân biệt khoảng trắng thừa
df_dedup['Desc_vs_Mapping_Match'] = (
    df_dedup['Description_Clean'].str.lower().str.strip()
    == df_dedup['Account_Name_Mapped'].str.lower().str.strip()
)

# Chỉ đánh dấu mismatch với những dòng có Account_Code mapped (bỏ unmapped ra)
df_dedup['Description_Mismatch'] = (
    ~df_dedup['Desc_vs_Mapping_Match'] & ~df_dedup['Is_Unmapped_Account']
)

# ---------- 4.6. Báo cáo đối chiếu ----------
print("=" * 70)
print("ĐỐI CHIẾU DESCRIPTION vs ACCOUNT_NAME TỪ MAPPING")
print("=" * 70)
print(f"Tổng dòng                              : {len(df_dedup)}")
print(f"Số dòng Description MATCH mapping      : {df_dedup['Desc_vs_Mapping_Match'].sum()}")
print(f"Số dòng Description MISMATCH mapping   : {df_dedup['Description_Mismatch'].sum()}")
print(f"Số dòng Unmapped (bỏ qua đối chiếu)    : {df_dedup['Is_Unmapped_Account'].sum()}")

# Xem chi tiết các cặp (Code, Description gốc, Description_Clean, Mapping name) bị mismatch
mismatch_detail = (
    df_dedup[df_dedup['Description_Mismatch']]
    [['Account_Code_Padded', 'Description_Original', 'Description_Clean', 'Account_Name_Mapped']]
    .drop_duplicates()
    .reset_index(drop=True)
)
print("\n--- CHI TIẾT CÁC DÒNG MISMATCH (unique) ---")
if len(mismatch_detail) == 0:
    print("  (Không có dòng nào mismatch — toàn bộ Description khớp mapping)")
else:
    print(mismatch_detail.to_string(index=False))

# Bảng đối chiếu tổng hợp theo Account_Code: mỗi mã desc có khớp mapping không
print("\n--- ĐỐI CHIẾU THEO TỪNG ACCOUNT_CODE ---")
compare_by_code = (
    df_dedup.groupby('Account_Code_Padded')
    .agg(
        So_dong=('Amount_GBP', 'size'),
        Desc_duy_nhat=('Description_Clean', lambda s: s.nunique()),
        Mapping_Name=('Account_Name_Mapped', 'first'),
        PnL_Category=('P&L_Category', 'first'),
        So_dong_match=('Desc_vs_Mapping_Match', 'sum'),
        So_dong_mismatch=('Description_Mismatch', 'sum'),
    )
    .reset_index()
)
compare_by_code['Trang_thai'] = np.where(
    compare_by_code['So_dong_mismatch'] == 0, "✅ OK", "⚠️ MISMATCH"
)
print(compare_by_code.to_string(index=False))

ĐỐI CHIẾU DESCRIPTION vs ACCOUNT_NAME TỪ MAPPING
Tổng dòng                              : 6805
Số dòng Description MATCH mapping      : 6700
Số dòng Description MISMATCH mapping   : 105
Số dòng Unmapped (bỏ qua đối chiếu)    : 0

--- CHI TIẾT CÁC DÒNG MISMATCH (unique) ---
Account_Code_Padded      Description_Original       Description_Clean        Account_Name_Mapped
               9000            Business Rates          Business Rates       Other Income/Expense
               5099    IT & Software Licences  It & Software Licences Cost of Goods Sold - Other
               6999    Marketing & Promotions  Marketing & Promotions   Other Operating Expenses
               6999     Employer NI & Pension   Employer Ni & Pension   Other Operating Expenses
               5099            Business Rates          Business Rates Cost of Goods Sold - Other
               9000             Property Rent           Property Rent       Other Income/Expense
               5099    Marketing & Promotions  

In [18]:
# ============================================================
# KHỐI 4 (FINAL): CHUẨN HÓA + RECLASS SUSPENSE + MAPPING CATEGORY
# Nguồn: df_dedup (từ Khối 3)
# ============================================================

# ---------- 4.1. Chuẩn hóa Account_Code ----------
def normalize_code(x):
    if pd.isna(x):
        return ''
    s = str(x).strip()
    if s.endswith('.0'):
        s = s[:-2]
    return s

df_dedup['Account_Code_Original'] = df_dedup['Account_Code'].apply(normalize_code)
df_dedup['Account_Code'] = df_dedup['Account_Code_Original'].str.zfill(4)

# ---------- 4.2. Mapping bảng chuẩn (Code → Name → Category) ----------
mapping_dict = {
    '4000': ('Food & Beverage Revenue',     'Revenue'),
    '4010': ('Retail Merchandise Revenue',  'Revenue'),
    '4020': ('Entertainment Revenue',       'Revenue'),
    '4030': ('Membership Revenue',          'Revenue'),
    '4040': ('Events & Hire Revenue',       'Revenue'),
    '5000': ('Cost of Goods Sold - F&B',    'COGS'),
    '5010': ('Cost of Goods Sold - Retail', 'COGS'),
    '6000': ('Wages & Salaries',            'OPEX_Wages'),
    '6010': ('Employer NI & Pension',       'OPEX_Wages'),
    '6100': ('Property Rent',               'OPEX_Fixed'),
    '6110': ('Business Rates',              'OPEX_Fixed'),
    '6200': ('Utilities - Electricity',     'OPEX_Variable'),
    '6210': ('Utilities - Gas/Water',       'OPEX_Variable'),
    '6300': ('Marketing & Promotions',      'OPEX_Variable'),
    '6400': ('Repairs & Maintenance',       'OPEX_Variable'),
    '6500': ('IT & Software Licences',      'OPEX_Fixed'),
    '6600': ('Insurance',                   'OPEX_Fixed'),
    '6700': ('Professional Fees',           'OPEX_Variable'),
    '7000': ('Depreciation',                'Non-Cash'),
    '7100': ('Bank Interest',               'Finance'),
}

SUSPENSE_ACCOUNTS = {'5099', '6999', '9000'}

# ---------- 4.3. Bảng reclass: Description → Account_Code đích ----------
# Key phải là lowercase để so khớp (không phân biệt hoa/thường)
reclass_by_description = {
    'food & beverage revenue':      '4000',
    'retail merchandise revenue':   '4010',
    'entertainment revenue':        '4020',
    'membership revenue':           '4030',
    'events & hire revenue':        '4040',
    'cost of goods sold - f&b':     '5000',
    'cost of goods sold - retail':  '5010',
    'wages & salaries':             '6000',
    'employer ni & pension':        '6010',
    'property rent':                '6100',
    'business rates':               '6110',
    'utilities - electricity':      '6200',
    'utilities - gas/water':        '6210',
    'marketing & promotions':       '6300',
    'repairs & maintenance':        '6400',
    'it & software licences':       '6500',
    'insurance':                    '6600',
    'professional fees':            '6700',
    'depreciation':                 '7000',
    'bank interest':                '7100',
}

# ---------- 4.4. Chuẩn hóa Description: lowercase → Title Case + fix acronym ----------
df_dedup['Description_Original'] = df_dedup['Description'].astype(str).str.strip()
df_dedup['Description_Clean'] = (
    df_dedup['Description_Original']
    .str.lower()
    .str.title()
    .str.strip()
)
# Fix acronym
df_dedup['Description_Clean'] = (
    df_dedup['Description_Clean']
    .str.replace(r'\bNi\b', 'NI', regex=True)
    .str.replace(r'\bIt\b', 'IT', regex=True)
)

# ---------- 4.5. RECLASS: ép suspense về account đúng theo Description ----------
def reclass_account(row):
    code = row['Account_Code']
    if code not in SUSPENSE_ACCOUNTS:
        return code   # giữ nguyên nếu không phải suspense
    # Tra Description (lowercase) trong bảng reclass
    key = row['Description_Original'].lower().strip()
    return reclass_by_description.get(key, code)  # không tìm thấy → giữ nguyên

df_dedup['Account_Code_Reclassed'] = df_dedup.apply(reclass_account, axis=1)
df_dedup['Was_Reclassed'] = (
    df_dedup['Account_Code'] != df_dedup['Account_Code_Reclassed']
)

# ---------- 4.6. Mapping Name + Category từ Account_Code SAU reclass ----------
df_dedup['Account_Name_Mapped'] = df_dedup['Account_Code_Reclassed'].map(
    lambda c: mapping_dict.get(c, ('UNMAPPED', 'UNMAPPED'))[0]
)
df_dedup['P&L_Category'] = df_dedup['Account_Code_Reclassed'].map(
    lambda c: mapping_dict.get(c, ('UNMAPPED', 'UNMAPPED'))[1]
)
df_dedup['Is_Unmapped_Account'] = df_dedup['P&L_Category'] == 'UNMAPPED'
df_dedup['Cannot_Reclass'] = (
    df_dedup['Account_Code'].isin(SUSPENSE_ACCOUNTS)
    & ~df_dedup['Was_Reclassed']
)

# ---------- 4.7. Đối chiếu Description vs Mapping (sau reclass) ----------
df_dedup['Desc_vs_Mapping_Match'] = (
    df_dedup['Description_Clean'].str.lower().str.strip()
    == df_dedup['Account_Name_Mapped'].str.lower().str.strip()
)
df_dedup['Description_Mismatch'] = (
    ~df_dedup['Desc_vs_Mapping_Match'] & ~df_dedup['Is_Unmapped_Account']
)

# ---------- 4.8. BÁO CÁO ----------
print("=" * 75)
print("KẾT QUẢ RECLASS SUSPENSE → BẢNG CHUẨN")
print("=" * 75)
print(f"Tổng dòng df_dedup                    : {len(df_dedup)}")
print(f"Số dòng bị reclass (đổi Account_Code) : {df_dedup['Was_Reclassed'].sum()}")
print(f"Số dòng không thể reclass (giữ nguyên): {df_dedup['Cannot_Reclass'].sum()}")
print(f"Unmapped sau reclass                  : {df_dedup['Is_Unmapped_Account'].sum()}")
print(f"Description mismatch sau reclass      : {df_dedup['Description_Mismatch'].sum()}")

# Chi tiết reclass: từ mã nào → mã nào
print("\n--- CHI TIẾT RECLASS (Từ → Đến) ---")
reclass_detail = (
    df_dedup[df_dedup['Was_Reclassed']]
    .groupby(['Account_Code', 'Account_Code_Reclassed', 'Account_Name_Mapped'])
    .size()
    .reset_index(name='So_dong')
)
print(reclass_detail.to_string(index=False))

# Các dòng không reclass được (nếu có) → cần review thủ công
if df_dedup['Cannot_Reclass'].sum() > 0:
    print("\n⚠️ CÁC DÒNG KHÔNG RECLASS ĐƯỢC (cần review thủ công):")
    print(
        df_dedup[df_dedup['Cannot_Reclass']]
        [['Txn_ID', 'Account_Code', 'Description_Original', 'Amount_GBP']]
        .drop_duplicates()
        .to_string(index=False)
    )
else:
    print("\n✅ Tất cả dòng suspense đều reclass được — không cần review thủ công.")

# Bảng đối chiếu cuối cùng theo Account_Code (SAU reclass)
print("\n--- ĐỐI CHIẾU CUỐI CÙNG THEO ACCOUNT_CODE ---")
compare_by_code = (
    df_dedup.groupby('Account_Code_Reclassed')
    .agg(
        So_dong=('Amount_GBP', 'size'),
        Mapping_Name=('Account_Name_Mapped', 'first'),
        PnL_Category=('P&L_Category', 'first'),
        Match=('Desc_vs_Mapping_Match', 'sum'),
        Mismatch=('Description_Mismatch', 'sum'),
    )
    .reset_index()
)
compare_by_code['Trang_thai'] = np.where(
    compare_by_code['Mismatch'] == 0, "✅ OK", "⚠️ MISMATCH"
)
print(compare_by_code.to_string(index=False))

KẾT QUẢ RECLASS SUSPENSE → BẢNG CHUẨN
Tổng dòng df_dedup                    : 6805
Số dòng bị reclass (đổi Account_Code) : 105
Số dòng không thể reclass (giữ nguyên): 0
Unmapped sau reclass                  : 0
Description mismatch sau reclass      : 0

--- CHI TIẾT RECLASS (Từ → Đến) ---
Account_Code Account_Code_Reclassed     Account_Name_Mapped  So_dong
        5099                   6000        Wages & Salaries        7
        5099                   6010   Employer NI & Pension        2
        5099                   6100           Property Rent        3
        5099                   6110          Business Rates        5
        5099                   6200 Utilities - Electricity        2
        5099                   6210   Utilities - Gas/Water        2
        5099                   6300  Marketing & Promotions        4
        5099                   6400   Repairs & Maintenance        2
        5099                   6500  IT & Software Licences        2
        5099        

In [20]:
# ============================================================
# KHỐI 5: DATA QUALITY REPORT
# Nguồn: df_dedup (đã reclass) + df_flagged (để đếm dup gốc)
# ============================================================

# Đếm các chỉ số từ 2 nguồn
blank_count     = blank_rows_count
dup_count       = len(df_flagged) - len(df_dedup)  # = 25
unmapped_count  = df_dedup['Is_Unmapped_Account'].sum()
suspense_orig   = df_dedup['Account_Code'].isin(SUSPENSE_ACCOUNTS).sum()
reclassed_count = df_dedup['Was_Reclassed'].sum()
cannot_reclass  = df_dedup['Cannot_Reclass'].sum()

# Missing Department: Revenue (4xxx) + COGS (5xxx) sau reclass
is_rev_cogs = df_dedup['Account_Code_Reclassed'].str.startswith(('4', '5'))
missing_dept_mask = (
    df_dedup['Department'].isna()
    | df_dedup['Department'].astype(str).str.strip().isin(
        ['', 'N/A', 'NA', '-', 'None', 'null', 'Unknown']
    )
)
df_dedup['Error_Warning'] = is_rev_cogs & missing_dept_mask
df_dedup['Warning_Reason'] = np.where(
    df_dedup['Error_Warning'], "Missing Department (Rev/COGS)", ""
)
missing_dept_count = df_dedup['Error_Warning'].sum()
rev_cogs_total     = is_rev_cogs.sum()
missing_dept_pct   = missing_dept_count / rev_cogs_total * 100 if rev_cogs_total else 0

# Informational: Description mismatch
df_dedup['Error_Info'] = df_dedup['Description_Mismatch']
df_dedup['Info_Reason'] = np.where(
    df_dedup['Error_Info'], "Description differs from CoA name", ""
)
info_count = df_dedup['Error_Info'].sum()

# Critical: Unmapped + Invalid date/amount + Sign violation + Cannot reclass
critical_unmapped  = df_dedup['Is_Unmapped_Account']
critical_bad_date  = df_dedup['Txn_Date'].isna()
critical_bad_amt   = df_dedup['Amount_GBP'].isna()
critical_sign      = df_dedup['Sign_Violation']
critical_cannot_re = df_dedup['Cannot_Reclass']

df_dedup['Error_Critical'] = (
    critical_unmapped | critical_bad_date | critical_bad_amt 
    | critical_sign | critical_cannot_re
)
df_dedup['Critical_Reason'] = np.select(
    [critical_unmapped, critical_cannot_re, critical_bad_date, critical_bad_amt, critical_sign],
    ["Unmapped Account", "Cannot Reclass Suspense", "Invalid Date",
     "Invalid Amount", "Wrong Sign"],
    default=""
)
critical_count = df_dedup['Error_Critical'].sum()

# Bảng DQ Report
dq_summary = pd.DataFrame({
    'Quality Test': [
        'Blank transaction rows',
        'Exact duplicate transactions (removed)',
        'Unmapped account entries (real)',
        'Suspense account entries (original)',
        'Suspense reclassed successfully',
        'Suspense cannot reclass (manual review)',
        'Missing department tags in Rev/COGS',
        'Description mismatch vs CoA',
        'Sign violations (wrong sign)',
        'Total Critical Errors',
        'Total Warning Errors',
        'Total Informational Errors',
    ],
    'Actual_Count': [
        blank_count,
        dup_count,
        unmapped_count,
        suspense_orig,
        reclassed_count,
        cannot_reclass,
        f"{missing_dept_count} / {rev_cogs_total} ({missing_dept_pct:.1f}%)",
        info_count,
        critical_sign.sum(),
        critical_count,
        missing_dept_count,
        info_count,
    ],
    'Status': [
        '✅ Matched baseline',
        '✅ Matched baseline',
        '✅ All mapped after reclass',
        '🟡 Pending classification',
        '✅ Reclassed',
        '✅ None' if cannot_reclass == 0 else '⚠️ Need review',
        '⚠️ Review required',
        'ℹ️ Info only',
        '✅ None' if critical_sign.sum() == 0 else '⚠️ Need review',
        '—', '—', '—'
    ]
})

print("=" * 90)
print("DATA QUALITY REPORT")
print("=" * 90)
print(dq_summary.to_string(index=False))

DATA QUALITY REPORT
                           Quality Test      Actual_Count                     Status
                 Blank transaction rows                30         ✅ Matched baseline
 Exact duplicate transactions (removed)                 0         ✅ Matched baseline
        Unmapped account entries (real)                 0 ✅ All mapped after reclass
    Suspense account entries (original)               105   🟡 Pending classification
        Suspense reclassed successfully               105                ✅ Reclassed
Suspense cannot reclass (manual review)                 0                     ✅ None
    Missing department tags in Rev/COGS 204 / 3165 (6.4%)         ⚠️ Review required
            Description mismatch vs CoA                 0               ℹ️ Info only
           Sign violations (wrong sign)                 0                     ✅ None
                  Total Critical Errors                 0                          —
                   Total Warning Errors      

In [21]:
# ============================================================
# KHỐI 5 (REVISED): CHUẨN HÓA DESCRIPTION BẰNG MODE THEO ACCOUNT_CODE
# + DATA QUALITY REPORT
# Nguồn: df_dedup (đã reclass từ Khối 4)
# ============================================================

# ---------- 5.1. Tính Description MODE cho mỗi Account_Code_Reclassed ----------
# Mode = giá trị xuất hiện nhiều nhất (sau khi đã Title Case + fix acronym)
desc_mode = (
    df_dedup.groupby('Account_Code_Reclassed')['Description_Clean']
    .agg(lambda s: s.mode().iloc[0] if len(s.mode()) > 0 else '')
    .to_dict()
)

print("=" * 75)
print("DESCRIPTION MODE THEO ACCOUNT_CODE (giá trị xuất hiện nhiều nhất)")
print("=" * 75)
mode_table = pd.DataFrame({
    'Account_Code': list(desc_mode.keys()),
    'Description_Mode': list(desc_mode.values()),
}).sort_values('Account_Code').reset_index(drop=True)
print(mode_table.to_string(index=False))

# ---------- 5.2. Fill Description_Final bằng mode ----------
df_dedup['Description_Mode'] = df_dedup['Account_Code_Reclassed'].map(desc_mode)
# Nếu một mã có mode rỗng (toàn bộ desc gốc rỗng) → fallback mapping name
df_dedup['Description_Final'] = df_dedup['Description_Mode'].where(
    df_dedup['Description_Mode'].astype(str).str.strip() != '',
    df_dedup['Account_Name_Mapped']
)

# ---------- 5.3. Đối chiếu Description_Clean vs Description_Final ----------
# Mismatch = dòng có Description gốc KHÁC với mode của account đó
df_dedup['Desc_vs_Mode_Match'] = (
    df_dedup['Description_Clean'].str.lower().str.strip()
    == df_dedup['Description_Final'].str.lower().str.strip()
)
df_dedup['Description_Outlier'] = ~df_dedup['Desc_vs_Mode_Match']

# ---------- 5.4. Missing Department ----------
is_rev_cogs = df_dedup['Account_Code_Reclassed'].str.startswith(('4', '5'))
missing_dept_mask = (
    df_dedup['Department'].isna()
    | df_dedup['Department'].astype(str).str.strip().isin(
        ['', 'N/A', 'NA', '-', 'None', 'null', 'Unknown']
    )
)
df_dedup['Error_Warning'] = is_rev_cogs & missing_dept_mask
df_dedup['Warning_Reason'] = np.where(
    df_dedup['Error_Warning'], "Missing Department (Rev/COGS)", ""
)
missing_dept_count = df_dedup['Error_Warning'].sum()
rev_cogs_total     = int(is_rev_cogs.sum())
missing_dept_pct   = missing_dept_count / rev_cogs_total * 100 if rev_cogs_total else 0

# ---------- 5.5. Critical (Unmapped + Invalid + Sign + Cannot Reclass) ----------
critical_unmapped  = df_dedup['Is_Unmapped_Account']
critical_bad_date  = df_dedup['Txn_Date'].isna()
critical_bad_amt   = df_dedup['Amount_GBP'].isna()
critical_sign      = df_dedup['Sign_Violation']
critical_cannot_re = df_dedup['Cannot_Reclass']

df_dedup['Error_Critical'] = (
    critical_unmapped | critical_bad_date | critical_bad_amt
    | critical_sign | critical_cannot_re
)
df_dedup['Critical_Reason'] = np.select(
    [critical_unmapped, critical_cannot_re, critical_bad_date,
     critical_bad_amt, critical_sign],
    ["Unmapped Account", "Cannot Reclass Suspense", "Invalid Date",
     "Invalid Amount", "Wrong Sign"],
    default=""
)
critical_count = int(df_dedup['Error_Critical'].sum())

# ---------- 5.6. Informational = Description outliers ----------
df_dedup['Error_Info'] = df_dedup['Description_Outlier']
df_dedup['Info_Reason'] = np.where(
    df_dedup['Error_Info'],
    "Description differs from account mode",
    ""
)
info_count = int(df_dedup['Error_Info'].sum())

# ---------- 5.7. DQ REPORT ----------
blank_count     = int(blank_rows_count)
dup_count       = int(len(df_flagged) - len(df_dedup))
unmapped_count  = int(df_dedup['Is_Unmapped_Account'].sum())
suspense_orig   = int(df_dedup['Account_Code'].isin(SUSPENSE_ACCOUNTS).sum())
reclassed_count = int(df_dedup['Was_Reclassed'].sum())
cannot_reclass  = int(df_dedup['Cannot_Reclass'].sum())
outlier_count   = int(df_dedup['Description_Outlier'].sum())

dq_summary = pd.DataFrame({
    'Quality Test': [
        'Blank transaction rows',
        'Exact duplicate transactions (removed)',
        'Unmapped account entries (real)',
        'Suspense account entries (original)',
        'Suspense reclassed successfully',
        'Suspense cannot reclass (manual review)',
        'Missing department tags in Rev/COGS',
        'Description outliers vs account mode',
        'Sign violations (wrong sign)',
        'Total Critical Errors',
        'Total Warning Errors',
        'Total Informational Errors',
    ],
    'Baseline_Expected': [
        30, 25, 105, 105, '—', '—', '204 / 3.181', '—', '—', '—', '—', '—'
    ],
    'Actual_Count': [
        blank_count,
        dup_count,
        unmapped_count,
        suspense_orig,
        reclassed_count,
        cannot_reclass,
        f"{missing_dept_count} / {rev_cogs_total} ({missing_dept_pct:.1f}%)",
        outlier_count,
        int(critical_sign.sum()),
        critical_count,
        missing_dept_count,
        info_count,
    ],
    'Status': [
        '✅ Matched baseline',
        '✅ Matched baseline',
        '✅ All mapped after reclass',
        '🟡 Pending classification',
        '✅ Reclassed',
        '✅ None' if cannot_reclass == 0 else '⚠️ Need review',
        '⚠️ Review required',
        'ℹ️ Info only',
        '✅ None' if critical_sign.sum() == 0 else '⚠️ Need review',
        '—', '—', '—'
    ]
})

print("\n" + "=" * 90)
print("DATA QUALITY REPORT")
print("=" * 90)
print(dq_summary.to_string(index=False))

# ---------- 5.8. Bảng đối chiếu từng mã ----------
print("\n" + "=" * 90)
print("ĐỐI CHIẾU DESCRIPTION THEO TỪNG ACCOUNT_CODE")
print("=" * 90)
compare_by_code = (
    df_dedup.groupby('Account_Code_Reclassed')
    .agg(
        So_dong=('Amount_GBP', 'size'),
        Description_Mode=('Description_Final', 'first'),
        PnL_Category=('P&L_Category', 'first'),
        So_desc_khac_nhau=('Description_Clean', 'nunique'),
        So_dong_outlier=('Description_Outlier', 'sum'),
    )
    .reset_index()
)
compare_by_code['Trang_thai'] = np.where(
    compare_by_code['So_dong_outlier'] == 0, "✅ OK",
    "⚠️ Có outlier (đã fill bằng mode)"
)
print(compare_by_code.to_string(index=False))

# ---------- 5.9. Chi tiết outlier (nếu có) ----------
outliers = df_dedup[df_dedup['Description_Outlier']]
if len(outliers) > 0:
    print("\n--- CHI TIẾT CÁC DÒNG OUTLIER (Description khác mode) ---")
    outlier_detail = (
        outliers.groupby(['Account_Code_Reclassed', 'Description_Clean', 'Description_Final'])
        .size().reset_index(name='So_dong')
        .sort_values(['Account_Code_Reclassed', 'So_dong'], ascending=[True, False])
    )
    print(outlier_detail.to_string(index=False))
else:
    print("\n✅ Không có outlier — mọi Description đã đồng nhất theo account.")

DESCRIPTION MODE THEO ACCOUNT_CODE (giá trị xuất hiện nhiều nhất)
Account_Code            Description_Mode
        4000     Food & Beverage Revenue
        4010  Retail Merchandise Revenue
        4020       Entertainment Revenue
        4030          Membership Revenue
        4040       Events & Hire Revenue
        5000    Cost Of Goods Sold - F&B
        5010 Cost Of Goods Sold - Retail
        6000            Wages & Salaries
        6010       Employer NI & Pension
        6100               Property Rent
        6110              Business Rates
        6200     Utilities - Electricity
        6210       Utilities - Gas/Water
        6300      Marketing & Promotions
        6400       Repairs & Maintenance
        6500      IT & Software Licences
        6600                   Insurance
        6700           Professional Fees
        7000                Depreciation
        7100               Bank Interest

DATA QUALITY REPORT
                           Quality Test Baseline_Exp

In [22]:
# ============================================================
# KHỐI 5B: IMPUTE MISSING DEPARTMENT THEO MODE TỪNG ACCOUNT_CODE
# Chỉ áp dụng cho Rev/COGS (4xxx, 5xxx)
# ============================================================

# ---------- 5B.1. Xác định mask missing dept ----------
df_dedup['Department_Clean'] = (
    df_dedup['Department']
    .astype(str).str.strip()
    .replace({'nan': '', 'None': '', 'N/A': '', 'NA': '', '-': '', 'null': '', 'Unknown': ''})
)

is_rev_cogs = df_dedup['Account_Code_Reclassed'].str.startswith(('4', '5'))
missing_dept = df_dedup['Department_Clean'] == ''
df_dedup['Missing_Dept_Original'] = is_rev_cogs & missing_dept

print("=" * 80)
print("BƯỚC 1: TRƯỚC KHI IMPUTE")
print("=" * 80)
print(f"Tổng dòng Rev/COGS           : {is_rev_cogs.sum()}")
print(f"Số dòng missing dept (Rev/COGS): {df_dedup['Missing_Dept_Original'].sum()}")
print(f"Tỷ lệ missing                : {df_dedup['Missing_Dept_Original'].sum() / is_rev_cogs.sum() * 100:.2f}%")

# ---------- 5B.2. Tính mode Department cho từng Account_Code ----------
# Chỉ tính mode từ những dòng CÓ department (không tính missing)
dept_mode_by_code = (
    df_dedup[df_dedup['Department_Clean'] != '']
    .groupby('Account_Code_Reclassed')['Department_Clean']
    .agg(lambda s: s.mode().iloc[0] if len(s.mode()) > 0 else None)
    .to_dict()
)

print("\n" + "=" * 80)
print("BƯỚC 2: MODE DEPARTMENT THEO TỪNG ACCOUNT_CODE")
print("=" * 80)
mode_df = pd.DataFrame([
    {'Account_Code': k, 'Mode_Department': v}
    for k, v in dept_mode_by_code.items()
])
print(mode_df.to_string(index=False))

# ---------- 5B.3. Impute ----------
def impute_dept(row):
    # Không phải Rev/COGS → không đụng
    if not row['Account_Code_Reclassed'].startswith(('4', '5')):
        return row['Department_Clean']
    # Đã có dept → giữ nguyên
    if row['Department_Clean'] != '':
        return row['Department_Clean']
    # Missing → lấy mode của account đó
    mode_val = dept_mode_by_code.get(row['Account_Code_Reclassed'], '')
    return mode_val if mode_val else 'UNASSIGNED'

df_dedup['Department_Imputed'] = df_dedup.apply(impute_dept, axis=1)

# Cờ đánh dấu dòng nào đã được impute
df_dedup['Was_Imputed_Dept'] = (
    df_dedup['Missing_Dept_Original']
    & (df_dedup['Department_Imputed'] != '')
    & (df_dedup['Department_Imputed'] != 'UNASSIGNED')
)

# Cờ đánh dấu dòng không thể impute (account đó không có dòng nào có dept)
df_dedup['Cannot_Impute_Dept'] = (
    df_dedup['Missing_Dept_Original']
    & (df_dedup['Department_Imputed'] == 'UNASSIGNED')
)

# ---------- 5B.4. Báo cáo ----------
print("\n" + "=" * 80)
print("BƯỚC 3: SAU KHI IMPUTE")
print("=" * 80)
print(f"Số dòng đã impute dept          : {df_dedup['Was_Imputed_Dept'].sum()}")
print(f"Số dòng không thể impute        : {df_dedup['Cannot_Impute_Dept'].sum()}")
print(f"Số dòng còn missing dept (Rev/COGS): {df_dedup['Cannot_Impute_Dept'].sum()}")

# Chi tiết impute theo Account_Code
impute_detail = (
    df_dedup[df_dedup['Was_Imputed_Dept']]
    .groupby(['Account_Code_Reclassed', 'Department_Imputed'])
    .agg(So_dong=('Amount_GBP', 'size'), Tong_GBP=('Amount_GBP', 'sum'))
    .reset_index()
)
print("\n--- CHI TIẾT IMPUTE THEO ACCOUNT_CODE ---")
print(impute_detail.to_string(index=False))

# Nếu có account không thể impute
if df_dedup['Cannot_Impute_Dept'].sum() > 0:
    print("\n⚠️ CÁC ACCOUNT KHÔNG THỂ IMPUTE (không có dòng nào có dept):")
    cannot_detail = (
        df_dedup[df_dedup['Cannot_Impute_Dept']]
        .groupby('Account_Code_Reclassed')
        .agg(So_dong=('Amount_GBP', 'size'), Tong_GBP=('Amount_GBP', 'sum'))
        .reset_index()
    )
    print(cannot_detail.to_string(index=False))
    print("\n→ Các dòng này cần review thủ công, không thể tự động impute.")
else:
    print("\n✅ Toàn bộ dòng missing dept đều impute được (mỗi account đều có ít nhất 1 dept mẫu).")

# ---------- 5B.5. Cập nhật Error_Warning sau impute ----------
# Warning chỉ còn những dòng KHÔNG impute được
df_dedup['Error_Warning'] = is_rev_cogs & df_dedup['Cannot_Impute_Dept']
df_dedup['Warning_Reason'] = np.where(
    df_dedup['Error_Warning'],
    "Missing Department - cannot impute (account has no dept samples)",
    ""
)
missing_dept_count = df_dedup['Error_Warning'].sum()
missing_dept_pct   = missing_dept_count / is_rev_cogs.sum() * 100 if is_rev_cogs.sum() else 0

print("\n" + "=" * 80)
print("TRẠNG THÁI WARNING SAU IMPUTE")
print("=" * 80)
print(f"Missing department tags in Rev/COGS: {missing_dept_count} / {is_rev_cogs.sum()} ({missing_dept_pct:.2f}%)")
print(f"→ Đã giảm từ {df_dedup['Missing_Dept_Original'].sum()} xuống {missing_dept_count}")

BƯỚC 1: TRƯỚC KHI IMPUTE
Tổng dòng Rev/COGS           : 3165
Số dòng missing dept (Rev/COGS): 204
Tỷ lệ missing                : 6.45%

BƯỚC 2: MODE DEPARTMENT THEO TỪNG ACCOUNT_CODE
Account_Code Mode_Department
        4000        FOOD_BEV
        4010      MEMBERSHIP
        4020          RETAIL
        4030        FOOD_BEV
        4040          EVENTS
        5000      MEMBERSHIP
        5010          RETAIL

BƯỚC 3: SAU KHI IMPUTE
Số dòng đã impute dept          : 204
Số dòng không thể impute        : 0
Số dòng còn missing dept (Rev/COGS): 0

--- CHI TIẾT IMPUTE THEO ACCOUNT_CODE ---
Account_Code_Reclassed Department_Imputed  So_dong   Tong_GBP
                  4000           FOOD_BEV       28  533848.57
                  4010         MEMBERSHIP       21  123450.18
                  4020             RETAIL       29  270883.17
                  4030           FOOD_BEV       36  117528.16
                  4040             EVENTS       38   87221.11
                  5000         ME

In [23]:
# ============================================================
# KHỐI 5 (CẬP NHẬT LẠI SAU IMPUTE)
# ============================================================

dq_summary = pd.DataFrame({
    'Quality Test': [
        'Blank transaction rows',
        'Exact duplicate transactions (removed)',
        'Unmapped account entries (real)',
        'Suspense account entries (original)',
        'Suspense reclassed successfully',
        'Suspense cannot reclass (manual review)',
        'Missing department tags in Rev/COGS (ORIGINAL)',
        'Missing department tags - IMPUTED by mode',
        'Missing department tags - CANNOT impute',
        'Description mismatch vs CoA',
        'Sign violations (wrong sign)',
        'Total Critical Errors',
        'Total Warning Errors (after impute)',
        'Total Informational Errors',
    ],
    'Actual_Count': [
        blank_count,
        dup_count,
        unmapped_count,
        suspense_orig,
        reclassed_count,
        cannot_reclass,
        f"{df_dedup['Missing_Dept_Original'].sum()} / {is_rev_cogs.sum()} "
        f"({df_dedup['Missing_Dept_Original'].sum()/is_rev_cogs.sum()*100:.1f}%)",
        df_dedup['Was_Imputed_Dept'].sum(),
        df_dedup['Cannot_Impute_Dept'].sum(),
        info_count,
        critical_sign.sum(),
        df_dedup['Error_Critical'].sum(),
        df_dedup['Error_Warning'].sum(),
        info_count,
    ],
    'Status': [
        '✅ Matched baseline',
        '✅ Matched baseline',
        '✅ All mapped after reclass',
        '🟡 Pending classification',
        '✅ Reclassed',
        '✅ None' if cannot_reclass == 0 else '⚠️ Need review',
        '⚠️ Matches baseline ~6.4%',
        '✅ Imputed (audit logged)',
        '✅ None' if df_dedup['Cannot_Impute_Dept'].sum() == 0 else '⚠️ Review',
        'ℹ️ Info only',
        '✅ None' if critical_sign.sum() == 0 else '⚠️ Review',
        '—', '—', '—'
    ]
})

print("=" * 95)
print("DATA QUALITY REPORT (SAU IMPUTE DEPARTMENT)")
print("=" * 95)
print(dq_summary.to_string(index=False))

DATA QUALITY REPORT (SAU IMPUTE DEPARTMENT)
                                  Quality Test      Actual_Count                     Status
                        Blank transaction rows                30         ✅ Matched baseline
        Exact duplicate transactions (removed)                 0         ✅ Matched baseline
               Unmapped account entries (real)                 0 ✅ All mapped after reclass
           Suspense account entries (original)               105   🟡 Pending classification
               Suspense reclassed successfully               105                ✅ Reclassed
       Suspense cannot reclass (manual review)                 0                     ✅ None
Missing department tags in Rev/COGS (ORIGINAL) 204 / 3165 (6.4%)  ⚠️ Matches baseline ~6.4%
     Missing department tags - IMPUTED by mode               204   ✅ Imputed (audit logged)
       Missing department tags - CANNOT impute                 0                     ✅ None
                   Description misma

In [25]:
# ============================================================
# KHỐI 6 (FULL): ADJUSTMENT LOG & RECONCILIATION
# Gồm 3 loại adjustment: Duplicate removed, Reclass suspense, Impute Department
# ============================================================

# ---------- 6.1. Adjustment Log ----------

# (a) Duplicate đã xóa
adj_duplicates = df_flagged[
    df_flagged.duplicated(keep='first')
].copy()
adj_duplicates['Adjustment_Type'] = 'Remove Duplicate'
adj_duplicates['From_Account']    = adj_duplicates['Account_Code']
adj_duplicates['To_Account']      = '—'
adj_duplicates['P&L_Impact_GBP']  = -adj_duplicates['Amount_GBP']
adj_duplicates['Reason']          = 'Exact duplicate removed'

# (b) Suspense đã reclass
adj_reclass = df_dedup[df_dedup['Was_Reclassed']].copy()
adj_reclass['Adjustment_Type'] = 'Reclass Suspense'
adj_reclass['From_Account']    = adj_reclass['Account_Code']
adj_reclass['To_Account']      = adj_reclass['Account_Code_Reclassed']
adj_reclass['P&L_Impact_GBP']  = 0  # chỉ đổi category, không đổi tổng
adj_reclass['Reason']          = 'Suspense account reclassed by Description'

# (c) Impute Department (từ Khối 5B)
adj_impute = df_dedup[df_dedup['Was_Imputed_Dept']].copy()
adj_impute['Adjustment_Type'] = 'Impute Department'
adj_impute['From_Account']    = adj_impute['Account_Code_Reclassed']
adj_impute['To_Account']      = adj_impute['Account_Code_Reclassed']  # không đổi code
adj_impute['P&L_Impact_GBP']  = 0  # chỉ điền dept, không đổi amount
adj_impute['Reason']          = (
    'Department imputed = mode of same Account_Code: '
    + adj_impute['Department_Imputed'].astype(str)
)

# Gộp tất cả vào adj_log
adj_log = pd.concat([
    adj_duplicates[['Txn_ID', 'Adjustment_Type', 'From_Account', 'To_Account',
                    'Amount_GBP', 'P&L_Impact_GBP', 'Reason']],
    adj_reclass[['Txn_ID', 'Adjustment_Type', 'From_Account', 'To_Account',
                 'Amount_GBP', 'P&L_Impact_GBP', 'Reason']],
    adj_impute[['Txn_ID', 'Adjustment_Type', 'From_Account', 'To_Account',
                'Amount_GBP', 'P&L_Impact_GBP', 'Reason']],
], ignore_index=True)

print("=" * 90)
print("ADJUSTMENT LOG SUMMARY")
print("=" * 90)
print(adj_log.groupby('Adjustment_Type').agg(
    So_dong=('Txn_ID', 'size'),
    Tong_Amount_GBP=('Amount_GBP', 'sum'),
    Tong_PnL_Impact=('P&L_Impact_GBP', 'sum'),
).reset_index().to_string(index=False))

# ---------- 6.2. Reconciliation ----------

# Chuẩn hóa code cho df_flagged
df_flagged['Account_Code_Norm'] = (
    df_flagged['Account_Code'].apply(normalize_code).str.zfill(4)
)

raw_total = df_flagged['Amount_GBP'].sum()

# Unmapped thực sự = code nằm ngoài mapping_dict
known_codes = set(mapping_dict.keys())
unmapped_mask_flagged = ~df_flagged['Account_Code_Norm'].isin(known_codes)
unmapped_val = df_flagged[unmapped_mask_flagged]['Amount_GBP'].sum()
mapped_val   = raw_total - unmapped_val

# Approved Adjustments = duplicate bị xóa (chỉ cái này ảnh hưởng Amount)
approved_dup_value = adj_duplicates['Amount_GBP'].sum()

clean_total = raw_total - approved_dup_value
dedup_total = df_dedup['Amount_GBP'].sum()

qa_checks = pd.DataFrame({
    'Reconciliation Check': [
        'Raw Total (sau bỏ blank, trước adj)',
        '  ├─ Mapped Total',
        '  ├─ Unmapped Total',
        'Check: Raw − (Mapped + Unmapped)  [=0]',
        'Approved Adjustments (duplicate removed)',
        'Clean Total (Raw − Approved Adj)',
        'Clean Total từ df_dedup (cross-check)',
        'Check: Clean vs df_dedup  [=0]',
    ],
    'Value_GBP': [
        raw_total,
        mapped_val,
        unmapped_val,
        raw_total - (mapped_val + unmapped_val),
        approved_dup_value,
        clean_total,
        dedup_total,
        clean_total - dedup_total,
    ]
})

print("\n" + "=" * 90)
print("RECONCILIATION CONTROL")
print("=" * 90)
print(qa_checks.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))

ADJUSTMENT LOG SUMMARY
  Adjustment_Type  So_dong  Tong_Amount_GBP  Tong_PnL_Impact
Impute Department      204        909716.22              0.0
 Reclass Suspense      105       -357590.04              0.0

RECONCILIATION CONTROL
                    Reconciliation Check    Value_GBP
     Raw Total (sau bỏ blank, trước adj) 3,922,672.74
                         ├─ Mapped Total 4,280,262.78
                       ├─ Unmapped Total  -357,590.04
  Check: Raw − (Mapped + Unmapped)  [=0]         0.00
Approved Adjustments (duplicate removed)         0.00
        Clean Total (Raw − Approved Adj) 3,922,672.74
   Clean Total từ df_dedup (cross-check) 3,922,672.74
          Check: Clean vs df_dedup  [=0]         0.00


In [27]:
# ============================================================
# XUẤT FILE DATA SẠCH: 2025_Actual_Clean.csv
# Nguồn: df_dedup (đã bỏ blank + xóa dup + reclass + impute dept)
# ============================================================

# Chọn các cột cần xuất (data sạch, bỏ cột tạm/audit)
output_cols = [
    'Txn_ID',
    'Txn_Date',
    'Period',
    'Account_Code_Reclassed',   # mã account sau reclass (dùng chính thức)
    'Account_Name_Mapped',      # tên account chuẩn
    'P&L_Category',             # phân loại Revenue/COGS/OPEX...
    'Description_Clean',        # description đã chuẩn hóa
    'Site_Code',
    'Department_Imputed',       # dept đã điền (đã xử lý missing)
    'Amount_GBP',
]

# Đổi tên cột cho gọn & chuẩn
df_clean_export = df_dedup[output_cols].copy()
df_clean_export = df_clean_export.rename(columns={
    'Account_Code_Reclassed': 'Account_Code',
    'Department_Imputed':     'Department',
})

# Xuất CSV
df_clean_export.to_csv('2025_Actual_Clean.csv', index=False, encoding='utf-8-sig')

print(f"✅ Đã xuất file: 2025_Actual_Clean.csv")
print(f"   Số dòng: {len(df_clean_export):,}")
print(f"   Số cột : {len(df_clean_export.columns)}")
print(f"   Cột    : {df_clean_export.columns.tolist()}")

✅ Đã xuất file: 2025_Actual_Clean.csv
   Số dòng: 6,805
   Số cột : 10
   Cột    : ['Txn_ID', 'Txn_Date', 'Period', 'Account_Code', 'Account_Name_Mapped', 'P&L_Category', 'Description_Clean', 'Site_Code', 'Department', 'Amount_GBP']
